# Task 5 — Create a Judge Model

An automated judge that grades generated descriptions using the same rubric from Task 1. The judge uses **Meta-Llama-3.1-8B-Instruct** (the model not used for generation in Task 2).

Latency and Cost are excluded from the judge — those are measured programmatically.

In [1]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

from shared.config import JUDGE_MODEL, NEBIUS_BASE_URL
from shared.constants import JUDGE_CRITERIA, JUDGE_SYSTEM_PROMPT, RUBRIC

print(f"Judge model: {JUDGE_MODEL}")

Judge model: meta-llama/Meta-Llama-3.1-8B-Instruct


## Judge Prompt

The prompt is built from the rubric definitions in `shared/constants.py`, so any rubric change automatically propagates to the judge. It covers the 5 judge criteria (Fluency, Grammar, Tone, Length, Grounding) with their good/ok/bad definitions.

For **Grounding**, the judge receives both the original product information and the generated description to check every claim against the source.

In [2]:
print(JUDGE_SYSTEM_PROMPT)

You are an evaluation judge for e-commerce product descriptions.

You will receive:
1. The original product information (name, attributes, material, warranty).
2. A generated product description to evaluate.

Rate the description on each of the following criteria using the rubric below.
For each criterion, first write a brief explanation of your reasoning, then give your verdict.

Rubric:

Fluency:
  good: Reads naturally with smooth sentence flow; no awkward phrasing. A native speaker would find nothing jarring.
  ok: Minor awkwardness — 1–2 clunky phrases — but the text is still easy to understand overall.
  bad: Choppy, hard to follow, or incoherent sentences. The reader has to re-read to understand the meaning.

Grammar:
  good: Zero spelling or punctuation errors.
  ok: 1–2 minor errors (e.g., a missing comma or minor typo) that do not change meaning or impede understanding.
  bad: 3 or more errors, OR any error that changes meaning or significantly impedes comprehension.

Tone:
 

## Pydantic Output Schema

Each criterion returns an `explanation` followed by a `verdict`. Pydantic enforces this structure via the API's structured output support.

### Why explanation comes before verdict

LLMs generate tokens left-to-right. If the verdict comes first, the model commits to a rating before articulating its reasoning, which can lead to post-hoc justification. By placing `explanation` first, the model is forced to reason through the evidence before deciding on a verdict — this is a form of chain-of-thought prompting baked into the output schema. It produces more consistent and better-calibrated ratings.

In [3]:
import json
from enum import Enum

from pydantic import BaseModel, Field


class Rating(str, Enum):
    good = "good"
    ok = "ok"
    bad = "bad"


class CriterionResult(BaseModel):
    explanation: str = Field(description="Reasoning for the verdict")
    verdict: Rating = Field(description="Rating: good, ok, or bad")


class JudgeOutput(BaseModel):
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult


print(json.dumps(JudgeOutput.model_json_schema(), indent=2))

{
  "$defs": {
    "CriterionResult": {
      "properties": {
        "explanation": {
          "description": "Reasoning for the verdict",
          "title": "Explanation",
          "type": "string"
        },
        "verdict": {
          "$ref": "#/$defs/Rating",
          "description": "Rating: good, ok, or bad"
        }
      },
      "required": [
        "explanation",
        "verdict"
      ],
      "title": "CriterionResult",
      "type": "object"
    },
    "Rating": {
      "enum": [
        "good",
        "ok",
        "bad"
      ],
      "title": "Rating",
      "type": "string"
    }
  },
  "properties": {
    "fluency": {
      "$ref": "#/$defs/CriterionResult"
    },
    "grammar": {
      "$ref": "#/$defs/CriterionResult"
    },
    "tone": {
      "$ref": "#/$defs/CriterionResult"
    },
    "length": {
      "$ref": "#/$defs/CriterionResult"
    },
    "grounding": {
      "$ref": "#/$defs/CriterionResult"
    }
  },
  "required": [
    "fluency",
    "gramm

## Judge Function

In [4]:
import time

from openai import OpenAI

client = OpenAI(
    base_url=NEBIUS_BASE_URL,
    api_key=os.environ.get("NEBIUS_API_KEY"),
)


def build_judge_user_message(product_info: dict, description: str) -> str:
    return (
        f"Product name: {product_info['product_name']}\n"
        f"Attributes: {product_info['Product_attribute_list']}\n"
        f"Material: {product_info['material']}\n"
        f"Warranty: {product_info['warranty']}\n\n"
        f"Generated description:\n{description}"
    )


def judge_description(
    product_info: dict,
    description: str,
    model: str = JUDGE_MODEL,
) -> tuple[JudgeOutput, float]:
    """Run the judge on a single description. Returns (parsed output, latency_ms)."""
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": build_judge_user_message(product_info, description)},
    ]

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "judge_output",
                "strict": True,
                "schema": JudgeOutput.model_json_schema(),
            },
        },
    )
    elapsed_ms = (time.perf_counter() - start) * 1000

    parsed = JudgeOutput.model_validate_json(response.choices[0].message.content)
    return parsed, elapsed_ms


print("Judge function ready.")

Judge function ready.


## Quick Test

Run the judge on a single product to verify the output schema and prompt work correctly.

In [5]:
import pandas as pd
from shared.config import OUTPUT_EXCEL

df = pd.read_excel(OUTPUT_EXCEL)

row = df.iloc[0]
product_info = {
    "product_name": row["product_name"],
    "Product_attribute_list": row["Product_attribute_list"],
    "material": row["material"],
    "warranty": row["warranty"],
}

result, latency = judge_description(product_info, row["generated_description"])
print(f"Product: {row['product_name']}")
print(f"Latency: {latency:.0f}ms")
print()

for criterion in JUDGE_CRITERIA:
    cr = getattr(result, criterion.lower())
    print(f"{criterion}: {cr.verdict.value}")
    print(f"  {cr.explanation}")
    print()

Product: Apple iPhone 15 Pro
Latency: 5663ms

Fluency: good
  The description reads naturally with smooth sentence flow, and the phrasing is clear and concise.

Grammar: good
  There are no spelling or punctuation errors in the description.

Tone: good
  The tone is friendly, persuasive, and professionally credible, maintaining a consistent sales voice throughout the description.

Length: good
  The description contains 76 words, which falls within the 50-90 word range.

Grounding: good
  The description accurately reflects the provided product information, including the A17 Pro chip, 120Hz ProMotion display, titanium frame, Ceramic Shield glass, and 1-year limited warranty. No fabricated details are present.

